In [1]:
#### Convert coffea output into root TH2's to feed into TUnfold
import coffea
import pickle
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import os
from src.plugins import addFiles
# import ROOT

%load_ext autoreload

%autoreload 2

In [2]:
fname = "coffeaOutput/dijet/ijetHists_fixSDmass_QCDsim_rap2.5_MG_JK2016APV.pkl"
with open(fname, "rb") as f:
    result = pickle.load( f )
print(result['ptreco_mreco_g'][{'syst':'nominal'}])

fname = "coffeaOutput/trijet/trijetHists_removeJMRJMS_QCDsim_rap2.5_herwig_allUncALLRespOnly.pkl"
with open(fname, "rb") as f:
    result = pickle.load( f )
print(result['misses_u'][{'syst':'nominal'}])

FileNotFoundError: [Errno 2] No such file or directory: 'coffeaOutput/dijet/ijetHists_fixSDmass_QCDsim_rap2.5_MG_JK2016APV.pkl'

In [ ]:

files = ["coffeaOutput/dijet/dijetHists_fixGenWeights_JetHT_2016APV.pkl", 
         "coffeaOutput/dijet/dijetHists_fixGenWeights_JetHT_2016.pkl", 
         "coffeaOutput/dijet/dijetHists_fixGenWeights_JetHT_2017.pkl", 
         "coffeaOutput/dijet/dijetHists_fixGenWeights_JetHT_2018.pkl" ]
combinedFile = addFiles(files)
if "ALL" in files[0]:
    year = "Run2UL"
elif "APV" in files[0]:
    year = "2016APV"
else:
    year = files[0][-8:-4]
print(combinedFile)
with open("coffeaOutput/dijet/ijetHists_fixSDmass_QCDsim_rap2.5_MG_JK"+year+".pkl", "wb") as f:
    pickle.dump( combinedFile, f)
fname = "coffeaOutput/dijet/ijetHists_fixSDmass_QCDsim_rap2.5_MG_JK"+year+".pkl"
with open(fname, "rb") as f:
    result = pickle.load( f )
availSysts = [ax for ax in result['ptreco_mreco_u'].project("syst").axes[0]]

In [3]:
def combineFiles(prepstr, IOV="ALL", RespOnly=False, inds=None):
    unc_srcs = ["nominal",'LuminosityDown', 'PUSFDown', 'Q2muRUp', 'L1prefiringUp', 'Q2muFUp', 'PDFUp', 'PDFDown', 'PUSFUp', 'LuminosityUp', 'L1prefiringDown', 'Q2muFDown', 'Q2muRDown', 'JERUp', 'JERDown', 'HEM', 'JES_AbsoluteMPFBiasUp', 'JES_AbsoluteMPFBiasDown', 'JES_AbsoluteScaleUp', 'JES_AbsoluteScaleDown', 'JES_AbsoluteStatUp', 'JES_AbsoluteStatDown', 'JES_FlavorQCDUp', 'JES_FlavorQCDDown', 'JES_FragmentationUp', 'JES_FragmentationDown', 'JES_PileUpDataMCUp', 'JES_PileUpDataMCDown', 'JES_PileUpPtBBUp', 'JES_PileUpPtBBDown', 'JES_PileUpPtEC1Up', 'JES_PileUpPtEC1Down', 'JES_PileUpPtEC2Up', 'JES_PileUpPtEC2Down', 'JES_PileUpPtHFUp', 'JES_PileUpPtHFDown', 'JES_PileUpPtRefUp', 'JES_PileUpPtRefDown', 'JES_RelativeFSRUp', 'JES_RelativeFSRDown', 'JES_RelativeJEREC1Up', 'JES_RelativeJEREC1Down', 'JES_RelativeJEREC2Up', 'JES_RelativeJEREC2Down', 'JES_RelativeJERHFUp', 'JES_RelativeJERHFDown', 'JES_RelativePtBBUp', 'JES_RelativePtBBDown', 'JES_RelativePtEC1Up', 'JES_RelativePtEC1Down', 'JES_RelativePtEC2Up', 'JES_RelativePtEC2Down', 'JES_RelativePtHFUp', 'JES_RelativePtHFDown', 'JES_RelativeBalUp', 'JES_RelativeBalDown', 'JES_RelativeSampleUp', 'JES_RelativeSampleDown', 'JES_RelativeStatECUp', 'JES_RelativeStatECDown', 'JES_RelativeStatFSRUp', 'JES_RelativeStatFSRDown', 'JES_RelativeStatHFUp', 'JES_RelativeStatHFDown', 'JES_SinglePionECALUp', 'JES_SinglePionECALDown', 'JES_SinglePionHCALUp', 'JES_SinglePionHCALDown', 'JES_TimePtEtaUp', 'JES_TimePtEtaDown', 'JMRUp', 'JMRDown', 'JMSUp', 'JMSDown']
    iovs = ["2016APV", "2016", "2017", "2018"]
    # unc_srcs = ["nominal","JER","JMR","JMS","HEM","AbsoluteMPFBias","AbsoluteScale","AbsoluteStat","FlavorQCD","Fragmentation","PileUpDataMC","PileUpPtBB","PileUpPtEC1","PileUpPtEC2"]
    files = []
    if IOV == "ALL" and inds==None:
        print("Adding years or not systs")
        for i in iovs:
            files.append(prepstr+i+".pkl")
    elif IOV == "ALL" and inds!=None:
        print("Adding sections")
        for i in iovs:
            for ind in inds:
                files.append(prepstr+i+".pkl")
    else:
        for src in unc_srcs:
            files.append(prepstr+src+IOV+".pkl")
    print("Final list of files ", files)
    combined = addFiles(files, RespOnly=RespOnly)
    outputFilename = prepstr+"allUnc"+IOV+".pkl"
    if RespOnly: outputFilename = prepstr+"allUnc"+IOV+"RespOnly.pkl"
    with open(outputFilename, "wb") as f:
        pickle.dump( combined, f)
    return combined
def combineSystFiles(prepstr, IOV, RespOnly=False):
    unc_srcs = ["nominal",'LuminosityDown', 'PUSFDown', 'Q2muRUp', 'L1prefiringUp', 'Q2muFUp', 'PDFUp', 'PDFDown', 'PUSFUp', 'LuminosityUp', 'L1prefiringDown', 'Q2muFDown', 'Q2muRDown', 'JERUp', 'JERDown', 'HEM', 'JES_AbsoluteMPFBiasUp', 'JES_AbsoluteMPFBiasDown', 'JES_AbsoluteScaleUp', 'JES_AbsoluteScaleDown', 'JES_AbsoluteStatUp', 'JES_AbsoluteStatDown', 'JES_FlavorQCDUp', 'JES_FlavorQCDDown', 'JES_FragmentationUp', 'JES_FragmentationDown', 'JES_PileUpDataMCUp', 'JES_PileUpDataMCDown', 'JES_PileUpPtBBUp', 'JES_PileUpPtBBDown', 'JES_PileUpPtEC1Up', 'JES_PileUpPtEC1Down', 'JES_PileUpPtEC2Up', 'JES_PileUpPtEC2Down', 'JES_PileUpPtHFUp', 'JES_PileUpPtHFDown', 'JES_PileUpPtRefUp', 'JES_PileUpPtRefDown', 'JES_RelativeFSRUp', 'JES_RelativeFSRDown', 'JES_RelativeJEREC1Up', 'JES_RelativeJEREC1Down', 'JES_RelativeJEREC2Up', 'JES_RelativeJEREC2Down', 'JES_RelativeJERHFUp', 'JES_RelativeJERHFDown', 'JES_RelativePtBBUp', 'JES_RelativePtBBDown', 'JES_RelativePtEC1Up', 'JES_RelativePtEC1Down', 'JES_RelativePtEC2Up', 'JES_RelativePtEC2Down', 'JES_RelativePtHFUp', 'JES_RelativePtHFDown', 'JES_RelativeBalUp', 'JES_RelativeBalDown', 'JES_RelativeSampleUp', 'JES_RelativeSampleDown', 'JES_RelativeStatECUp', 'JES_RelativeStatECDown', 'JES_RelativeStatFSRUp', 'JES_RelativeStatFSRDown', 'JES_RelativeStatHFUp', 'JES_RelativeStatHFDown', 'JES_SinglePionECALUp', 'JES_SinglePionECALDown', 'JES_SinglePionHCALUp', 'JES_SinglePionHCALDown', 'JES_TimePtEtaUp', 'JES_TimePtEtaDown', 'JMRUp', 'JMRDown', 'JMSUp', 'JMSDown']
    iovs = ["2016APV", "2016", "2017", "2018"]
    # unc_srcs = ["nominal","JER","JMR","JMS","HEM","AbsoluteMPFBias","AbsoluteScale","AbsoluteStat","FlavorQCD","Fragmentation","PileUpDataMC","PileUpPtBB","PileUpPtEC1","PileUpPtEC2"]
    files = []
    if IOV == "ALL":
        print("Adding years not systs")
        for i in iovs:
            files.append(prepstr+i+".pkl")
    else:
        for src in unc_srcs:
            files.append(prepstr+src+IOV+".pkl")
    print("Final list of files ", files)
    combined = addFiles(files, RespOnly=RespOnly)
    print("Combined file hists", combined.keys())
    outputFilename = prepstr+"allUnc"+IOV+".pkl"
    print("output filename ", outputFilename)
    if RespOnly: outputFilename = prepstr+"allUnc"+IOV+"RespOnly.pkl"
    with open(outputFilename, "wb") as f:
        pickle.dump( combined, f)
    return combined

In [5]:
# combineSystFiles("coffeaOutput/trijet/trijetHistsTest_wXSscaling_QCDsim_pt200.0_rapidity2.5_", "2018")

#combineSystFiles("coffeaOutput/dijet/dijetHists_wXSscaling_QCDsim_pt200.0_rapidity2.5_", "2018", RespOnly=False)
combineSystFiles("coffeaOutput/trijet/trijetHists_fixGenWeights_QCD_MG_", "ALL", RespOnly=True)



Adding years not systs
Final list of files  ['coffeaOutput/trijet/trijetHists_fixGenWeights_QCD_MG_2016APV.pkl', 'coffeaOutput/trijet/trijetHists_fixGenWeights_QCD_MG_2016.pkl', 'coffeaOutput/trijet/trijetHists_fixGenWeights_QCD_MG_2017.pkl', 'coffeaOutput/trijet/trijetHists_fixGenWeights_QCD_MG_2018.pkl']
dict_keys(['misses_u', 'misses_g', 'fakes_u', 'fakes_g', 'ptreco_mreco_u', 'ptreco_mreco_g', 'rho_reco_u', 'rho_reco_g', 'ptgen_mgen_u', 'ptgen_mgen_g', 'response_rho_u', 'response_rho_g', 'response_matrix_u', 'response_matrix_g', 'cutflow', 'jkflow', 'alljet_ptreco_mreco', 'btag_eta', 'MET_over_sumET_pt_reco', 'MET_pt_reco', 'HT_nocuts', 'HT_wXS', 'HT_aftercuts', 'dphimin_gen', 'dphimin_reco', 'asymm_reco', 'asymm_gen', 'mass_orig', 'sdmass_orig', 'sdmass_ak8corr', 'sdmass_ak4corr', 'jet_eta_phi_precuts', 'jet_eta_phi_preveto', 'jet_pt_eta_phi', 'fakes_eta_phi', 'fakes_asymm_dphi', 'ptreco_mreco_fine_u', 'ptreco_mreco_fine_g'])
starting file  coffeaOutput/trijet/trijetHists_fixGenWe

{'response_matrix_u': Hist(
   StrCategory(['pythiaMG2016APV', 'pythiaMG2016', 'pythiaMG2017', 'pythiaMG2018'], growth=True, name='dataset', label='Primary dataset'),
   StrCategory(['nominal', 'LuminosityDown', 'PUSFDown', 'Q2muRUp', 'L1prefiringUp', 'Q2muFUp', 'PDFUp', 'PDFDown', 'PUSFUp', 'LuminosityUp', 'L1prefiringDown', 'Q2muFDown', 'Q2muRDown', 'JERUp', 'JERDown', 'HEM', 'JES_AbsoluteMPFBiasUp', 'JES_AbsoluteMPFBiasDown', 'JES_AbsoluteScaleUp', 'JES_AbsoluteScaleDown', 'JES_AbsoluteStatUp', 'JES_AbsoluteStatDown', 'JES_FlavorQCDUp', 'JES_FlavorQCDDown', 'JES_FragmentationUp', 'JES_FragmentationDown', 'JES_PileUpDataMCUp', 'JES_PileUpDataMCDown', 'JES_PileUpPtBBUp', 'JES_PileUpPtBBDown', 'JES_PileUpPtEC1Up', 'JES_PileUpPtEC1Down', 'JES_PileUpPtEC2Up', 'JES_PileUpPtEC2Down', 'JES_PileUpPtHFUp', 'JES_PileUpPtHFDown', 'JES_PileUpPtRefUp', 'JES_PileUpPtRefDown', 'JES_RelativeFSRUp', 'JES_RelativeFSRDown', 'JES_RelativeJEREC1Up', 'JES_RelativeJEREC1Down', 'JES_RelativeJEREC2Up', 'JES_

In [ ]:
fname = "coffeaOutput/trijetHists_wXSscaling_QCDsim_pt200.0rapidity2.5_bloosejesjecALL.pkl"
with open(fname, "rb") as f:
    result = pickle.load( f )
print("values: ", result['response_matrix_u'].project("ptgen", "mgen").values())
print("variances: ", result['response_matrix_u'].project("ptgen", "mgen").variances())

In [ ]:
%matplotlib inline
# fname = "coffeaOutput/dijetHists_wXSscaling_QCDsim_pt200.0_rapidity2.5jesjecALL.pkl"
#fname = "coffeaOutput/trijetHists_wXSscaling_QCDsim_pt200.0rapidity2.5_bloosejesjecALL.pkl" <-- has reco hist bug
fname = "coffeaOutput/dijetHists_wXSscaling_QCDsim_pt200.0_rapidity2.5jesjecL1PU2016.pkl"
# fname ="coffeaOutput/trijetHistsTest_JetHT_pt200.0_eta2.4_bbloose.pkl"
eras = ["2016", "2017", "2018", "2016APV"]
dir = "rootFiles/"
for year in eras:
    print(year)
    if year in fname:
        print(year)
        year_str = year
        break
    else:
        year_str = "ALL"
with open(fname, "rb") as f:
    result = pickle.load( f )
print([ax for ax in result['jet_pt_mass_reco_u'][{'dataset':sum}].axes[0]])
axis_names = [ax for ax in result['jet_pt_mass_reco_u'][{'dataset':sum}].axes[0]]
print(axis_names)
cats = [cat for cat in result['jet_pt_mass_reco_u'][{'ptreco':sum, 'dataset':sum, 'mreco':sum}].axes[0]]
print(cats)
if "JetHT" in fname:
    if "dijet" in  fname:
        rootfname = 'dijetHistsJetHT_jec_' + year_str + '.root'
    elif "trijet" in fname:
        rootfname = 'trijetHistsJetHT_jec_' + year_str + '.root'
elif "QCD" in fname:
    if "dijet" in  fname:
        rootfname = 'dijetHistsQCDsim_jec_' + year_str + '.root'
    # integrate and sum over axes
    elif "trijet" in fname:
        rootfname = 'trijetHistsQCDsim_jec' + year_str + '.root'
if os.path.exists(dir + rootfname):
    rootfile = uproot.recreate(dir+rootfname)
else:
    rootfile = uproot.create(dir+rootfname)
    #### only add hists containing gen if SIM files
response_entries = {}
ptgen_mgen_entries = {}
ptreco_mreco_entries = {}
fakes_entries = {}
misses_entries = {}
if "QCD" in fname:
    for syst in cats:
        rootfile['ptgen_mgen_u_'+syst] = result['response_matrix_u'][{'syst':syst}].project("ptgen", "mgen")
        rootfile['ptgen_mgen_g_'+syst] = result['response_matrix_g'][{'syst':syst}].project("ptgen","mgen")
        rootfile['ptreco_mreco_u_'+syst] = result['response_matrix_u'][{'syst':syst}].project("ptreco", "mreco")
        rootfile['ptreco_mreco_g_'+syst] = result['response_matrix_g'][{'syst':syst}].project("ptreco","mreco")
        rootfile['misses_ptgen_mgen_'+syst] = result['misses'][{'syst':syst}].project("ptgen", "mgen")
        rootfile['fakes_ptreco_mreco_'+syst] = result['fakes'][{'syst':syst}].project("ptreco","mreco")
        response_matrix_u_values, ptreco_edges, mreco_edges, ptgen_edges, mgen_edges = result['response_matrix_u'][{'syst':syst}].project("ptreco", "mreco", "ptgen", "mgen").to_numpy(flow=True)
        response_matrix_g_values, ptreco_edges, mreco_edges, ptgen_edges, mgen_edges= result['response_matrix_g'][{'syst':syst}].project("ptreco", "mreco", "ptgen", "mgen").to_numpy(flow=True)
        print(response_matrix_g_values.shape)
        ptreco_centers = (ptreco_edges[:-1]+ptreco_edges[1:])/2
        print(ptreco_centers[np.newaxis])
        mreco_centers = (mreco_edges[:-1]+mreco_edges[1:])/2
        ptgen_centers = (ptgen_edges[:-1]+ptgen_edges[1:])/2
        mgen_centers = (mgen_edges[:-1]+mgen_edges[1:])/2
        print(np.shape(response_matrix_g_values))
        print(np.shape(response_matrix_g_values[np.newaxis]))
        response_entries.update({'groomed_'+syst:response_matrix_g_values[np.newaxis],
                           'ungroomed_'+syst:response_matrix_u_values[np.newaxis]})
    rootfile['response']=response_entries
else:
    for syst in cats:
        rootfile['ptreco_mreco_u_'+syst] = result['jet_pt_mass_reco_u'][{'syst':syst}].project("ptreco", "mreco")
        rootfile['ptreco_mreco_g_'+syst] = result['jet_pt_mass_reco_g'][{'syst':syst}].project("ptreco","mreco")
rootfile.close()

In [ ]:
import uproot
fname = "coffeaOutput/dijetHists_wXSscaling_QCDsim_pt200.0_rapidity2.5jesjecL1PU2016.pkl"
with open(fname, "rb") as f:
    result = pickle.load( f )
print(result.keys())
response_matrix_u_values, ptreco_edges, mreco_edges, ptgen_edges, mgen_edges = result['response_matrix_u'][{'syst':syst}].project("ptreco", "mreco", "ptgen", "mgen").to_numpy(flow=True)
print(response_matrix_g_values.shape)
response_matrix_g_values= result['response_matrix_g'][{'syst':'nominal'}].project("ptreco", "mreco", "ptgen", "mgen").values()
nptreco,nmassreco,nptgen,nmassgen = response_matrix_g_values.shape
response_matrix_g_final = response_matrix_g_values.reshape( (nptreco)*(nmassreco), (nptgen)*(nmassgen) ) 
response_matrix_u_final = response_matrix_u_values.reshape( (nptreco+2)*(nmassreco+2), (nptgen+2)*(nmassgen+2) ) 
print(response_matrix_g_final.shape)
response_matrix_g_final2 = result['response_matrix_g'][{'syst':"nominal"}].project("ptreco", "mreco", "ptgen", "mgen").values(flow=True).reshape( (nptreco+2)*(nmassreco+2), (nptgen+2)*(nmassgen+2))
ptreco_mreco_values= result['jet_pt_mass_reco_g'][{'syst':'nominal'}].project("ptreco", "mreco").values(flow=True)
print(ptreco_mreco_values.shape)
print(ptreco_mreco_values[1][0])
print(np.sum(response_matrix_g_final[24,:]))
# print(np.sum(response_matrix_g_final2[26,:]))
# print(np.sum(response_matrix_u_final[26,:]))

In [ ]:
# fname = uproot.open('root://cmsxrootd.fnal.gov//store/mc/RunIISummer20UL18NanoAODv9/QCD_Pt_1400to1800_TuneCP5_13TeV_pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/280000/42B4DEA0-C0E7-A944-80DC-1D107A0F63FB.root')

In [ ]:
# print(fname["Events"].keys("*HTXS*"))

In [ ]:
hists = uproot.open('rootFiles/dijetHistsQCDsim_jec_2016.root')
print(hists.keys())
hist_vals_g, hist_edges, hist_widths = hists['ptreco_mreco_g_nominal'].to_numpy(flow=False)
hist_vals_u, hist_edges, hist_widths = hists['ptreco_mreco_u_nominal'].to_numpy(flow=False)
print(hist_vals_u.shape)
print(hist_vals_g[1][0])
print(hist_vals_u[1][0])

In [ ]:
import uproot
hists = uproot.open(rootFiles/dijetHistsQCDsim_jec_2016.root)
print(hists['response'].keys())
resp_groomed = hists['response']['groomed_nominal'].array().to_numpy()[0].reshape((nptreco)*(nmassreco), (nptgen)*(nmassgen))
print(np.sum(resp_groomed[24,:]))
plt.figure(figsize = (20,20))
plt.imshow( resp_groomed, vmax=10, aspect="equal", cmap="Blues" )
plt.xlabel("GEN", fontsize=10)
plt.ylabel("RECO", fontsize=10)
plt.tick_params(labelsize=20)